In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
print(tf.__version__)

2.21.0


In [2]:
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2026-10-26 13:00:20--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349.84K  --.-KB/s    in 0.03s   

2026-10-26 13:00:20 (11.1 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2026-10-26 13:00:20--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv’

valid-data.tsv      100%[==============

In [3]:
train_df = pd.read_csv(train_file_path, sep="\t", header=None, names=["label", "message"])
test_df = pd.read_csv(test_file_path, sep="\t", header=None, names=["label", "message"])

train_df["y"] = (train_df["label"] == "spam").astype("float32")
test_df["y"] = (test_df["label"] == "spam").astype("float32")

MAX_TOKENS = 5000
SEQ_LEN = 50

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN,
    standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(train_df["message"].values)

X_train = vectorizer(train_df["message"].values).numpy()
X_test = vectorizer(test_df["message"].values).numpy()
y_train = train_df["y"].values
y_test = test_df["y"].values

model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,), dtype="int64"),
    layers.Embedding(MAX_TOKENS, 32),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.4),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

n_spam = y_train.sum()
n_ham = len(y_train) - n_spam
class_weight = {0: 1.0, 1: float(n_ham / n_spam) ** 0.5}

model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=2,
)

def predict_message(pred_text):
    x = vectorizer(np.array([pred_text])).numpy()
    p = float(model.predict(x, verbose=0)[0][0])
    return [p, "spam" if p >= 0.5 else "ham"]

print(predict_message("how are you doing today?"))

Epoch 1/15
131/131 - 5s - 41ms/step - accuracy: 0.9124 - loss: 0.4220 - val_accuracy: 0.9713 - val_loss: 0.0926
Epoch 2/15
131/131 - 3s - 21ms/step - accuracy: 0.9844 - loss: 0.1114 - val_accuracy: 0.9835 - val_loss: 0.0587
Epoch 3/15
131/131 - 3s - 26ms/step - accuracy: 0.9895 - loss: 0.0665 - val_accuracy: 0.9813 - val_loss: 0.0726
Epoch 4/15
131/131 - 3s - 21ms/step - accuracy: 0.9966 - loss: 0.0283 - val_accuracy: 0.9784 - val_loss: 0.0688
Epoch 5/15
131/131 - 3s - 21ms/step - accuracy: 0.9986 - loss: 0.0124 - val_accuracy: 0.9828 - val_loss: 0.0743
[0.00808822363615036, 'ham']


In [4]:
def test_predictions():
    test_messages = [
        "how are you doing today",
        "sale today! to stop texts call 98912460324",
        "i dont want to go. can we try it a different day? available sat",
        "our new mobile video service is live. just install on your phone to start watching.",
        "you have won £1000 cash! call to claim your prize.",
        "i'll bring it tomorrow. don't forget the milk.",
        "wow, is your arm alright. that happened to me one time too",
    ]
    test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
    passed = True
    for msg, ans in zip(test_messages, test_answers):
        prediction = predict_message(msg)
        if prediction[1] != ans:
            passed = False
    print("You passed the challenge. Great job!" if passed else "You haven't passed yet. Keep trying.")

test_predictions()

You passed the challenge. Great job!
